# 改进的练习
我用Deepseek将四个python代码模块中重要的部分挖去，重新填写

# data_fetcher

In [2]:
import yfinance as yf
import os

def get_data(ticker, start, end, proxy='http://127.0.0.1:7897'):
    """获取股票数据"""
    # ========== 你的代码在这里 ==========
    # 1. 如果 proxy 存在，设置环境变量 HTTP_PROXY 和 HTTPS_PROXY
    if proxy:
        os.environ['HTTP_PROXY'] = proxy
        os.environ['HTTPS_PROXY'] = proxy

    # 2. 用 yf.download() 下载数据
    data = yf.download(ticker = ticker, start = start, end= end)

    # 3. 如果列名是多层（nlevels > 1），用下划线展平列名
    # 提示：'_'.join(col).strip()
    if data.columns.nlevels > 1:
        data.columns = ['_'.join(col).strip() for col in data.columns.values]

    # 4. 统一列名为 Open, High, Low, Close, Volume
    # 提示：遍历 data.columns，根据 'Open' in col 等条件构建 rename_dict
    rename_dict = {}
    for col in data.columns:
        if 'Open' in col:
            rename_dic[col] = 'Open'
        elif 'High' in col:
            rename_dict[col] = 'High'
        elif 'Low' in col:
            rename_dict[col] = 'Low'
        elif 'Close' in col:
            rename_dict[col] = 'Close'
        elif 'Volume' in col:
            rename_dict[col] = 'Volume'
    data = data.rename(rename_dict)




    # 5. 返回 data

    # ==================================
    return data

# strategy

In [3]:
import pandas as pd

def add_ma(data, fast=5, slow=20):
    """添加移动平均线"""
    # ========== 你的代码在这里 ==========
    # 计算 MA5（快速均线）
    data['MA5'] = data['Close'].rolling(window=fast).mean()

    # 计算 MA20（慢速均线）
    data['MA20'] = data['Close'].rolling(window=slow).mean()

    # ==================================
    return data


def generate_signals(data):
    """生成买卖信号"""
    # ========== 你的代码在这里 ==========
    # 1. 初始化 Signal 列，默认值为 0
    data['Signal'] = 0

    # 2. 当 MA5 > MA20 时，将 Signal 设为 1
    data.loc[data['MA5'] > data['MA20'],'Signal'] = 1

    # 3. 计算 Position 列（Signal 的差值）
    data['Position'] = data['Signal'].diff()

    # ==================================
    return data

# backtest

In [4]:
def calculate_returns(data):
    """计算策略收益"""
    # ========== 你的代码在这里 ==========
    # 1. 计算日收益率 Returns（用 Close 列的 pct_change）
    data['Returns'] = data['Close'].pct_change()

    # 2. 计算策略收益率 Strategy_Returns（Signal 向后移一天，乘以 Returns）
    # 提示：用 shift(1)
    data['Strategy_Returns'] = data['Signal'].shift(1) * data['Returns']

    # 3. 计算策略累积净值 Cumulative_Strategy
    # 提示：(1 + Strategy_Returns) 的 cumprod
    # 注意列名是 'Strategy_Returns'（不是 data['Strategy_Returns']）
    data['Cumulative_Strategy'] = (1 + data['Strategy_Returns']).cumprod()

    # 4. 计算市场累积净值 Cumulative_Market（买入持有基准）
    # 提示：(1 + Returns) 的 cumprod
    data['Cumulative_Market'] = (1 + data['Returns']).cumprod()

    # ==================================
    return data


def sharpe_ratio(returns, rf=0):
    """计算夏普比率"""
    # ========== 你的代码在这里 ==========
    # 1. 删除 returns 中的缺失值
    daily_returns = returns.dropna()
    # 2. 计算年化夏普比率
    # 公式：(平均日收益率 - 无风险利率) / 日收益率标准差 * sqrt(252)
    # 提示：returns.mean(), returns.std(), 252 ** 0.5
    result = (daily_returns.mean() - rf) / daily_returns.std() * (252 ** 0.5)

    # ==================================
    return result  # 注意：这里应该返回计算出的数值

# main

In [5]:
# ========== 你的代码在这里 ==========
# 1. 从 data_fetcher 导入 get_data
# 2. 从 strategy 导入 add_ma, generate_signals
# 3. 从 backtest 导入 calculate_returns, sharpe_ratio
from data_fetcher import get_data
from strategy import add_ma, generate_signals
from backtest import calculate_returns, sharpe_ratio

# 4. 用 get_data 获取 AAPL 2024-01-01 到 2024-12-31 的数据
# 股票代码：'AAPL'，开始：'2024-01-01'，结束：'2024-12-31'
data = get_data('AAPL','2024-01-01','2024-12-31')

# 5. 依次调用 add_ma, generate_signals, calculate_returns
data = add_ma(data)
data = generate_signals(data)
data = calculate_returns(data)


# 6. 打印夏普比率（保留3位小数）
# 提示：print(f"... {sharpe_ratio(...):.3f}")
print(f"... {sharpe_ratio(data['Strategy_Returns']):.3f}")


# ==================================

ModuleNotFoundError: No module named 'data_fetcher'